<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes./blob/main/Modulo_de_Funciones_requeridas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from typing import List, Tuple, Dict, Optional


# MODULO 1: Validacion y Coherencia Dimensional
def validar_matrices(cargas: List[List[float]],
                      capacidades: List[List[float]]) -> bool:
    if not cargas or not capacidades:
        return False

    n_cargas = len(cargas)
    n_capacidades = len(capacidades)

    if n_cargas != n_capacidades or n_cargas < 2:
        return False

    m = len(cargas[0])
    if m < 2:
        return False

    for fila in cargas:
        if len(fila) != m:
            return False

    for fila in capacidades:
        if len(fila) != m:
            return False

    for fila in cargas:
        for peso in fila:
            if peso < 0:
                return False

    for fila in capacidades:
        for capacidad in fila:
            if capacidad <= 0:
                return False

    return True


# MODULO 2: Calculo de Ocupacion y Deteccion de Sobrecarga
def calcular_ocupacion(cargas: List[List[float]],
                        capacidades: List[List[float]]) -> Dict:
    filas = len(cargas)
    columnas = len(cargas[0])

    matriz_porcentajes = []
    celdas_sobrecargadas = []

    for i in range(filas):
        fila_porcentajes = []
        for j in range(columnas):
            porcentaje = (cargas[i][j] / capacidades[i][j]) * 100.0
            fila_porcentajes.append(porcentaje)
            if porcentaje > 100.0:
                celdas_sobrecargadas.append((i, j))
        matriz_porcentajes.append(fila_porcentajes)

    return {
        "matriz_porcentajes": matriz_porcentajes,
        "celdas_sobrecargadas": celdas_sobrecargadas,
    }


# MODULO 3: Evaluacion de Balance y Simetria
def evaluar_balance(cargas: List[List[float]], tolerancia: float) -> Dict:
    filas = len(cargas)
    columnas = len(cargas[0])

    pesos_por_fila = [sum(fila) for fila in cargas]

    mitad = columnas // 2
    if columnas % 2 == 0:
        columnas_izquierda = range(0, mitad)
        columnas_derecha = range(mitad, columnas)
    else:
        # columna del medio se omite (eje de simetria de la aeronave)
        columnas_izquierda = range(0, mitad)
        columnas_derecha = range(mitad + 1, columnas)

    suma_izquierda = sum(
        cargas[i][j] for i in range(filas) for j in columnas_izquierda
    )
    suma_derecha = sum(
        cargas[i][j] for i in range(filas) for j in columnas_derecha
    )

    desbalance_lateral = abs(suma_izquierda - suma_derecha)
    balance_aprobado = desbalance_lateral <= tolerancia

    return {
        "pesos_por_fila": pesos_por_fila,
        "desbalance_lateral": desbalance_lateral,
        "balance_aprobado": balance_aprobado,
    }


# MODULO 4: Extraccion de Submatriz de Sobrecarga Critica
def extraer_submatriz_critica(matriz_porcentajes: List[List[float]],
                               k: int, p: int) -> Optional[List[List[float]]]:
    filas = len(matriz_porcentajes)
    columnas = len(matriz_porcentajes[0]) if filas > 0 else 0

    if k <= 0 or p <= 0 or k > filas or p > columnas:
        return None

    mejor_promedio = float("-inf")
    mejor_conteo = -1
    mejor_submatriz = None

    for i in range(filas - k + 1):
        for j in range(columnas - p + 1):
            submatriz = [fila[j:j + p] for fila in matriz_porcentajes[i:i + k]]
            valores = [valor for fila in submatriz for valor in fila]

            promedio = sum(valores) / len(valores)
            conteo_sobrecarga = sum(1 for v in valores if v > 100.0)

            if promedio > mejor_promedio or (promedio == mejor_promedio and conteo_sobrecarga > mejor_conteo):
                mejor_promedio = promedio
                mejor_conteo = conteo_sobrecarga
                mejor_submatriz = submatriz

    return mejor_submatriz